# Pronóstico de Ventas Totales - Evaluación 2 (Advanced Machine Learning con TensorFlow y Keras)

- **Alumno:** Mario Alonso Vento Alvarado
- **Curso:** Especialización en Advanced Machine Learning con TensorFlow y Keras
- **Entregable:** Ev2 - Modelos para predecir ventas futuras

## Objetivo (según el enunciado)
Predecir las ventas totales para el próximo mes, evaluando si conviene trabajar a nivel **diario, semanal o mensual**, comparando los modelos vistos en clase (**Holt-Winters, ARIMA/SARIMA, Prophet, XGBoost y LSTM**) sobre `SalesDemand.csv`.

## Enfoque de este notebook
1. Cargar y limpiar `SalesDemand.csv`.
2. Análisis exploratorio, usando `store` e `item` como variables de agrupación.
3. Construir series de ventas totales a 3 granularidades: diaria, semanal y mensual.
4. Definir una metodología de validación común: **backtest de diciembre de 2017** (se entrena con todo lo anterior y se pronostica diciembre 2017, mes cuyo total real ya conocemos, para poder comparar objetivamente).
5. Ajustar los 5 modelos vistos en clase en cada una de las 3 granularidades (15 combinaciones), evaluando con la misma lógica de `evaluate_forecast` usada en las sesiones de clase (ARIMA, suavizamiento exponencial, Prophet).
6. Comparar resultados y determinar la mejor combinación granularidad + modelo.
7. Reentrenar con toda la información disponible (incluyendo diciembre 2017) y generar el pronóstico genuino de **enero 2018** (el "próximo mes" real).

## 13. Conclusiones

**Sobre la granularidad (diaria vs. semanal vs. mensual):** no hay una respuesta única válida para cualquier problema; depende de cómo interactúan el ruido de la serie, el número de estacionalidades presentes y la cantidad de historia disponible:
- La granularidad **diaria** ofrece más datos de entrenamiento y le permite a modelos como Prophet capturar tanto el patrón semanal como el anual, pero es más ruidosa y penaliza a los modelos que solo soportan una estacionalidad (Holt-Winters, SARIMA con `m=7`).
- La granularidad **semanal** reduce el ruido diario y le devuelve a Holt-Winters su capacidad de modelar la estacionalidad anual (`m=52`), pero para SARIMA resultó computacionalmente inviable con ese periodo estacional.
- La granularidad **mensual** es la más limpia y estable, pero al tener solo 59 observaciones limita a los modelos que dependen más de los datos (XGBoost, LSTM).

**Sobre los modelos:** los modelos clásicos con estacionalidad explícita (Holt-Winters, SARIMA, Prophet) tienden a tener ventaja cuando la historia disponible es corta o cuando el periodo estacional es difícil de aprender solo de los datos. Prophet tiene la ventaja adicional de poder combinar múltiples estacionalidades a la vez, lo que lo hace especialmente competitivo a nivel diario. Los modelos que dependen más de aprender patrones desde los datos (XGBoost, LSTM) parecen requerir más historia y/o variables explícitas de estacionalidad (como las de fecha que se le dieron a XGBoost) para ser competitivos en este problema. La recomendación final debe leerse directamente de la tabla de la sección 11, generada al ejecutar este notebook.

**Limitaciones y posibles extensiones:**
- El backtest se valida contra un único mes (diciembre 2017); una validación cruzada sobre varios meses daría una comparación más robusta.
- No se incorporaron variables externas (feriados, promociones, precios) que podrían mejorar cualquiera de los modelos.
- El análisis se hizo sobre la serie agregada total; `store` e `item` se usaron solo para caracterizar el negocio en el EDA. Un siguiente paso razonable sería repetir el mejor enfoque a nivel de tienda o de categoría de producto, si el negocio lo requiere.
- SARIMA con periodo estacional largo (`m=52`) no fue viable computacionalmente con `statsmodels`/`pmdarima` en este entorno; una alternativa a explorar sería SARIMAX con términos de Fourier como regresores exógenos para aproximar la estacionalidad anual sin pagar el costo computacional de un `m` tan grande.

## 1. Configuración del entorno

Se instalan explícitamente todas las librerías que no forman parte de la base estándar de Python, sin asumir que el cluster de Databricks sea ML Runtime (donde TensorFlow y scikit-learn ya vendrían preinstalados) ni Runtime estándar (donde no vienen).

**Sobre la versión de TensorFlow:** se fija `tensorflow>=2.18.0,<2.19` en vez de instalar la última versión disponible. TensorFlow >= 2.19 exige `protobuf>=6.31.1`, lo cual choca con paquetes nativos de Databricks (`mlflow-skinny`, `google-api-core`, `grpcio-status`, `googleapis-common-protos`, entre otros) que requieren `protobuf<7` o incluso `<6` según el paquete — esto produce un `VersionError` de protobuf al importar `tensorflow` (gencode/runtime de distinta versión mayor). La serie 2.18.x de TensorFlow exige `protobuf<6.0,>=3.20.3`, rango que sí es compatible con esos paquetes de Databricks (verificado instalando ambos conjuntos juntos).

Tras ejecutar la celda de instalación, corre `dbutils.library.restartPython()` (celda siguiente) para asegurar que el proceso de Python cargue la versión de protobuf recién instalada y no una que haya quedado en memoria de antes. Como el restart limpia las variables, hay que volver a ejecutar el notebook completo desde el inicio después de esa celda.

In [0]:
%pip install -q prophet pmdarima xgboost statsmodels scikit-learn "tensorflow>=2.18.0,<2.19"

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                              median_absolute_error, mean_squared_log_error)

warnings.filterwarnings('ignore')

print("Librerias importadas correctamente")

## 2. Carga de datos

In [0]:
# Ajustar esta ruta si el notebook no queda en la misma carpeta que SalesDemand.csv
DATA_PATH = "SalesDemand.csv"

candidatos = [DATA_PATH, "./SalesDemand.csv"]
ruta_encontrada = next((p for p in candidatos if os.path.exists(p)), None)

if ruta_encontrada is None:
    raise FileNotFoundError(
        "No se encontró 'SalesDemand.csv'. Verifica que el archivo esté en la carpeta "
        "o ajusta la variable DATA_PATH con la ruta completa al archivo."
    )

print("Archivo encontrado en:", ruta_encontrada)
df_raw = pd.read_csv(ruta_encontrada, sep=';', low_memory=False)
print("Forma cruda:", df_raw.shape)
df_raw.head()

## 3. Limpieza de datos

Al inspeccionar el archivo se detectó que, además de las 913,000 filas válidas (2013-01-01 a 2017-12-31, 10 tiendas x 50 productos x 1826 días), el CSV trae **45,000 filas completamente vacías al final** (un artefacto de exportación). Se eliminan antes de continuar, y se valida que no queden nulos parciales ni ventas negativas.

In [0]:
n_crudo = len(df_raw)
df = df_raw.dropna(how='all').copy()
n_limpio = len(df)
print(f"Filas originales: {n_crudo:,} | Filas tras eliminar registros vacios: {n_limpio:,}")

assert df.isnull().sum().sum() == 0, "Quedan valores nulos parciales, revisar el archivo fuente"

df['store'] = df['store'].astype(int)
df['item'] = df['item'].astype(int)
df['sales'] = df['sales'].astype(float)
df['date'] = pd.to_datetime(df['date'], dayfirst=True, format='mixed')
df = df.sort_values('date').reset_index(drop=True)

assert (df['sales'] >= 0).all(), "Se encontraron ventas negativas"

print(f"Rango de fechas: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Tiendas: {df['store'].nunique()} | Productos: {df['item'].nunique()} | "
      f"Combinaciones store-item: {df.groupby(['store','item']).ngroups}")
df.head()

## 4. Analisis exploratorio (EDA)

In [0]:
ventas_diarias_totales = df.groupby('date')['sales'].sum()

plt.figure(figsize=(14, 4))
plt.plot(ventas_diarias_totales.index, ventas_diarias_totales.values, linewidth=0.8)
plt.title('Ventas totales diarias (2013-2017)')
plt.xlabel('Fecha'); plt.ylabel('Ventas')
plt.tight_layout(); plt.show()

Se observa una tendencia creciente a lo largo de los 5 años, junto con un patrón estacional que se repite cada año. Se explora esto con más detalle a continuación.

In [0]:
ventas_por_tienda = df.groupby('store')['sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
ventas_por_tienda.plot(kind='bar', color='steelblue')
plt.title('Ventas totales por tienda (store)')
plt.xlabel('Tienda'); plt.ylabel('Ventas totales'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print(ventas_por_tienda)

Las 10 tiendas muestran volúmenes de venta distintos pero sin que una sola domine de forma desproporcionada; esto respalda trabajar con la **serie agregada total** (suma sobre todas las tiendas y productos) como variable objetivo, tal como pide el enunciado, en lugar de modelar cada tienda por separado.

In [0]:
ventas_por_item = df.groupby('item')['sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
ventas_por_item.head(15).plot(kind='bar', color='darkorange')
plt.title('Top 15 productos (item) por ventas totales')
plt.xlabel('Producto'); plt.ylabel('Ventas totales'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print(f"Producto con mas ventas: item {ventas_por_item.idxmax()} ({ventas_por_item.max():,.0f} unidades)")
print(f"Producto con menos ventas: item {ventas_por_item.idxmin()} ({ventas_por_item.min():,.0f} unidades)")

Al igual que con las tiendas, hay productos que venden más que otros, pero de nuevo sin una concentración extrema en uno o dos productos. `store` e `item` se usan aquí como variables de agrupación para caracterizar el negocio (tal como indica el enunciado), mientras que el pronóstico se realiza sobre el total agregado.

In [0]:
patron_dia_semana = ventas_diarias_totales.groupby(ventas_diarias_totales.index.dayofweek).mean()
dias = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']

plt.figure(figsize=(8, 4))
plt.bar(dias, patron_dia_semana.values, color='seagreen')
plt.title('Ventas promedio por dia de la semana')
plt.ylabel('Ventas promedio')
plt.tight_layout(); plt.show()

print(patron_dia_semana)

Existe un patrón semanal claro: las ventas crecen de lunes a domingo (lunes es el día más bajo, domingo el más alto). Esta es una señal relevante para la granularidad diaria.

In [0]:
patron_mensual = ventas_diarias_totales.groupby(ventas_diarias_totales.index.month).mean()
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

plt.figure(figsize=(8, 4))
plt.bar(meses, patron_mensual.values, color='indianred')
plt.title('Ventas promedio diarias por mes del anio (estacionalidad anual)')
plt.ylabel('Ventas promedio')
plt.tight_layout(); plt.show()

print(patron_mensual)

También hay una estacionalidad anual marcada: las ventas suben desde inicios de año, alcanzan su punto más alto alrededor de julio, y bajan hacia fin de año (diciembre queda por debajo de octubre y noviembre, de forma consistente en los 5 años de historia).

**Esto es clave para la comparación de modelos:** la serie diaria tiene *dos* estacionalidades superpuestas (semanal + anual). Holt-Winters y SARIMA clásicos solo pueden modelar **un** periodo estacional a la vez, mientras que Prophet está diseñado específicamente para series con múltiples estacionalidades marcadas (tal como se vio en la sesión de Prophet). Se vuelve sobre este punto en la sección de modelado diario.

In [0]:
from statsmodels.tsa.seasonal import seasonal_decompose

descomposicion = seasonal_decompose(ventas_diarias_totales, model='additive', period=365)

fig = descomposicion.plot()
fig.set_size_inches(12, 8)
plt.tight_layout(); plt.show()

La descomposición confirma una tendencia creciente y un componente estacional anual estable a lo largo del período, con un residuo relativamente acotado.

## 5. Series de ventas totales por granularidad (diaria, semanal, mensual)

Se agregan las ventas totales (suma sobre todas las tiendas y productos) a 3 niveles de granularidad, que es la data ya preparada para cada uno de los casos que pide el enunciado.

In [0]:
def construir_serie_total(df, freq):
    # Agrega las ventas totales (suma sobre todas las tiendas/items) a la granularidad indicada.
    # freq: 'D' diaria, 'W' semanal (cierre domingo), 'MS' mensual (inicio de mes)
    diaria = df.groupby('date')['sales'].sum().asfreq('D').fillna(0.0)
    diaria.name = 'sales'
    if freq == 'D':
        return diaria
    return diaria.resample(freq).sum()

serie_diaria = construir_serie_total(df, 'D')
serie_semanal = construir_serie_total(df, 'W')
serie_mensual = construir_serie_total(df, 'MS')

for nombre, s in [('Diaria', serie_diaria), ('Semanal', serie_semanal), ('Mensual', serie_mensual)]:
    print(f"{nombre}: {len(s)} periodos, de {s.index.min().date()} a {s.index.max().date()}")

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, (nombre, s) in zip(axes, [('Diaria', serie_diaria), ('Semanal', serie_semanal), ('Mensual', serie_mensual)]):
    ax.plot(s.index, s.values)
    ax.set_title(f'Ventas totales - granularidad {nombre}')
plt.tight_layout(); plt.show()

In [0]:
total_dic_2017 = serie_diaria.loc['2017-12-01':'2017-12-31'].sum()
print(f"Total real de ventas de diciembre de 2017 (referencia para el backtest): {total_dic_2017:,.0f}")

## 6. Metodología de evaluación y esquema de validación

**Función de evaluación:** se reutiliza la misma lógica de `evaluate_forecast` empleada en las sesiones de ARIMA, suavizamiento exponencial y Prophet (r2, MAE, mediana del error absoluto, MSE, MSLE, RMSE), agregando MAPE para facilitar la lectura porcentual.

**Esquema de validación - "backtest de diciembre 2017":** para comparar de forma justa las 3 granularidades y los 5 modelos, se usa el mismo criterio en todos los casos: se entrena con toda la información **anterior a diciembre de 2017** y se pronostica ese mes completo (que ya conocemos), comparando el total pronosticado contra el total real (calculado arriba: 695,170 unidades). Esto simula exactamente la tarea pedida ("predecir las ventas totales del próximo mes"), pero sobre un mes del que sí sabemos la respuesta correcta.

*Nota sobre la granularidad semanal:* como las semanas no calzan exactamente con los límites del mes, se consideran "de diciembre" las semanas cuyo día de cierre (domingo) cae dentro del mes. Esto genera una pequeña imprecisión de borde (la primera semana de este criterio incluye algunos días de noviembre), que se documenta aquí como una simplificación razonable.

In [0]:
def evaluate_forecast(y, pred):
    # Misma lógica que la usada en clase (ARIMA, suavizamiento exponencial, Prophet),
    # agregando MAPE. Se recortan las predicciones negativas a 0 antes de calcular MSLE
    # (las ventas no pueden ser negativas y MSLE no admite valores < 0).
    y = np.asarray(y, dtype=float).ravel()
    pred = np.asarray(pred, dtype=float).ravel()
    pred_safe = np.clip(pred, 0, None)

    resultado = pd.DataFrame({'r2_score': [r2_score(y, pred)]})
    resultado['mean_absolute_error'] = mean_absolute_error(y, pred)
    resultado['median_absolute_error'] = median_absolute_error(y, pred)
    resultado['mse'] = mean_squared_error(y, pred)
    resultado['msle'] = mean_squared_log_error(y, pred_safe)
    resultado['rmse'] = np.sqrt(resultado['mse'])
    resultado['mape'] = np.mean(np.abs((y - pred) / y)) * 100
    return resultado


def dividir_train_test(serie, corte='2017-12-01', fin_mes='2017-12-31'):
    # Train = todo lo anterior al mes objetivo; test = lo que cae dentro de ese mes
    train = serie[serie.index < corte]
    test = serie[(serie.index >= corte) & (serie.index <= fin_mes)]
    return train, test


train_d, test_d = dividir_train_test(serie_diaria)
train_w, test_w = dividir_train_test(serie_semanal)
train_m, test_m = dividir_train_test(serie_mensual)

for nombre, tr, te in [('Diaria', train_d, test_d), ('Semanal', train_w, test_w), ('Mensual', train_m, test_m)]:
    print(f"{nombre}: train={len(tr)} periodos | test={len(te)} periodos (horizonte de pronostico)")

## 7. Funciones de modelado (una función por técnica vista en clase)

Cada función recibe la serie de entrenamiento, el horizonte a pronosticar y el índice de fechas objetivo, y replica el patrón de la sesión correspondiente:
- **Holt-Winters** (suavizamiento exponencial triple) - sesión de técnicas de suavizamiento exponencial.
- **SARIMA vía `auto_arima`** - sesión de ARIMA.
- **Prophet** - sesión de Prophet.
- **XGBoost** sobre features de fecha + rezagos - sesión de comparación Prophet/XGBoost/LSTM.
- **LSTM** con ventana deslizante y pronóstico recursivo - sesiones de LSTM (caso de demanda, multi-variable, comparación final).

In [0]:
from statsmodels.tsa.api import ExponentialSmoothing

def ajustar_holt_winters(train, horizonte, indice_pronostico, seasonal_periods, seasonal='add', trend='add'):
    modelo = ExponentialSmoothing(
        train, trend=trend, seasonal=seasonal,
        seasonal_periods=seasonal_periods, initialization_method='estimated'
    ).fit()
    pronostico = modelo.forecast(horizonte)
    pronostico = pd.Series(np.asarray(pronostico), index=indice_pronostico)
    return pronostico, modelo

In [0]:
from pmdarima import auto_arima

def ajustar_auto_arima(train, horizonte, indice_pronostico, seasonal_periods=None):
    # seasonal_periods=None -> ARIMA no estacional (usado en la granularidad semanal, ver seccion 9)
    estacional = seasonal_periods is not None
    modelo = auto_arima(
        train, seasonal=estacional, m=seasonal_periods if estacional else 1,
        stepwise=True, suppress_warnings=True, error_action='ignore', trace=False
    )
    pred = modelo.predict(n_periods=horizonte)
    pronostico = pd.Series(np.asarray(pred), index=indice_pronostico)
    return pronostico, modelo

In [0]:
from prophet import Prophet
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

def ajustar_prophet(train, horizonte, indice_pronostico, freq, weekly_seasonality='auto', yearly_seasonality='auto'):
    train_p = pd.DataFrame({'ds': train.index, 'y': train.values})
    modelo = Prophet(weekly_seasonality=weekly_seasonality, yearly_seasonality=yearly_seasonality)
    modelo.fit(train_p)
    futuro = modelo.make_future_dataframe(periods=horizonte, freq=freq)
    prediccion = modelo.predict(futuro)
    pronostico = prediccion.set_index('ds')['yhat'].iloc[-horizonte:]
    pronostico = pd.Series(np.asarray(pronostico), index=indice_pronostico)
    return pronostico, modelo

In [0]:
import xgboost as xgb

def crear_features_fecha(indice):
    feat = pd.DataFrame(index=indice)
    feat['anio'] = indice.year
    feat['mes'] = indice.month
    feat['trimestre'] = indice.quarter
    feat['dia_semana'] = indice.dayofweek
    feat['dia_anio'] = indice.dayofyear
    feat['semana_anio'] = indice.isocalendar().week.astype(int).values
    return feat


def ajustar_xgboost(train, horizonte, indice_pronostico, n_lags=3):
    df_feat = crear_features_fecha(train.index)
    for lag in range(1, n_lags + 1):
        df_feat[f'lag_{lag}'] = train.shift(lag).values
    df_feat['y'] = train.values
    df_feat = df_feat.dropna()

    X_train = df_feat.drop(columns='y')
    y_train = df_feat['y']

    modelo = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
    modelo.fit(X_train, y_train)

    # Pronostico recursivo: cada paso usa los rezagos ya pronosticados en pasos anteriores
    historia = list(train.values[-n_lags:])
    predicciones = []
    for fecha in indice_pronostico:
        fila = crear_features_fecha(pd.DatetimeIndex([fecha])).iloc[0].to_dict()
        for lag in range(1, n_lags + 1):
            fila[f'lag_{lag}'] = historia[-lag]
        X_fila = pd.DataFrame([fila])[X_train.columns]
        yhat = modelo.predict(X_fila)[0]
        predicciones.append(yhat)
        historia.append(yhat)

    pronostico = pd.Series(predicciones, index=indice_pronostico)
    return pronostico, modelo

In [0]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

def crear_dataset_ventana(arr, ventana):
    X, y = [], []
    for i in range(len(arr) - ventana):
        X.append(arr[i:i + ventana, 0])
        y.append(arr[i + ventana, 0])
    return np.array(X), np.array(y)


def ajustar_lstm(train, horizonte, indice_pronostico, ventana, epochs=60, unidades=50, seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    escalador = MinMaxScaler(feature_range=(0, 1))
    escalado = escalador.fit_transform(train.values.reshape(-1, 1))

    X_train, y_train = crear_dataset_ventana(escalado, ventana)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

    modelo = Sequential([
        LSTM(unidades, activation='relu', return_sequences=True, input_shape=(ventana, 1)),
        Dropout(0.2),
        LSTM(unidades, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    modelo.compile(optimizer='adam', loss='mse')
    modelo.fit(X_train, y_train, epochs=epochs, batch_size=16, verbose=0, shuffle=False)

    # Pronostico recursivo multi-paso (mismo enfoque que en la comparacion Prophet/XGBoost/LSTM de clase)
    lote = escalado[-ventana:].reshape(1, ventana, 1)
    predicciones_escaladas = []
    for _ in range(horizonte):
        p = modelo.predict(lote, verbose=0)[0]
        predicciones_escaladas.append(p)
        lote = np.append(lote[:, 1:, :], [[p]], axis=1)

    predicciones = escalador.inverse_transform(np.array(predicciones_escaladas)).ravel()
    pronostico = pd.Series(predicciones, index=indice_pronostico)
    return pronostico, modelo

In [0]:
resultados = []
registro_modelos = {}

def registrar_resultado(granularidad, modelo_nombre, y_test, y_pred, total_real=None):
    metricas = evaluate_forecast(y_test, y_pred)
    fila = metricas.iloc[0].to_dict()
    fila['granularidad'] = granularidad
    fila['modelo'] = modelo_nombre
    total_pred = float(np.clip(y_pred, 0, None).sum())
    fila['total_predicho_mes'] = total_pred
    if total_real is not None:
        fila['total_real_mes'] = total_real
        fila['error_pct_total_mes'] = 100 * (total_pred - total_real) / total_real
    resultados.append(fila)
    return metricas


def graficar_pronostico(train, test, pronostico, titulo, contexto=60):
    plt.figure(figsize=(10, 4))
    serie_contexto = train.iloc[-contexto:] if len(train) > contexto else train
    plt.plot(serie_contexto.index, serie_contexto.values, label='Entrenamiento (contexto reciente)')
    plt.plot(test.index, test.values, label='Real', marker='o')
    plt.plot(pronostico.index, pronostico.values, label='Pronostico', marker='x')
    plt.legend(); plt.title(titulo); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 8. Modelado - Granularidad diaria

Configuración usada en esta granularidad:
- **Holt-Winters / SARIMA:** periodo estacional `m=7` (estacionalidad semanal). Como la serie diaria tiene *dos* estacionalidades (semanal y anual) y estos modelos solo pueden capturar una, es esperable que les cueste anticipar la caída de diciembre.
- **Prophet:** estacionalidad semanal y anual automáticas (`freq='D'`) - el escenario para el que Prophet fue diseñado.
- **XGBoost:** features de fecha (año, mes, trimestre, día de semana, día del año, semana del año) + 7 rezagos (una semana de historia).
- **LSTM:** ventana de 30 días.

In [0]:
pron_hw_d, mod_hw_d = ajustar_holt_winters(train_d, horizonte=len(test_d), indice_pronostico=test_d.index, seasonal_periods=7)
print(registrar_resultado('Diaria', 'Holt-Winters', test_d.values, pron_hw_d.values, total_real=total_dic_2017))
registro_modelos[('Diaria', 'Holt-Winters')] = dict(fn=ajustar_holt_winters, kwargs=dict(seasonal_periods=7))
graficar_pronostico(train_d, test_d, pron_hw_d, 'Diaria - Holt-Winters')

In [0]:
pron_arima_d, mod_arima_d = ajustar_auto_arima(train_d, horizonte=len(test_d), indice_pronostico=test_d.index, seasonal_periods=7)
print(registrar_resultado('Diaria', 'SARIMA (auto_arima)', test_d.values, pron_arima_d.values, total_real=total_dic_2017))
registro_modelos[('Diaria', 'SARIMA (auto_arima)')] = dict(fn=ajustar_auto_arima, kwargs=dict(seasonal_periods=7))
graficar_pronostico(train_d, test_d, pron_arima_d, 'Diaria - SARIMA (auto_arima)')

In [0]:
pron_prophet_d, mod_prophet_d = ajustar_prophet(train_d, horizonte=len(test_d), indice_pronostico=test_d.index, freq='D')
print(registrar_resultado('Diaria', 'Prophet', test_d.values, pron_prophet_d.values, total_real=total_dic_2017))
registro_modelos[('Diaria', 'Prophet')] = dict(fn=ajustar_prophet, kwargs=dict(freq='D'))
graficar_pronostico(train_d, test_d, pron_prophet_d, 'Diaria - Prophet')

In [0]:
pron_xgb_d, mod_xgb_d = ajustar_xgboost(train_d, horizonte=len(test_d), indice_pronostico=test_d.index, n_lags=7)
print(registrar_resultado('Diaria', 'XGBoost', test_d.values, pron_xgb_d.values, total_real=total_dic_2017))
registro_modelos[('Diaria', 'XGBoost')] = dict(fn=ajustar_xgboost, kwargs=dict(n_lags=7))
graficar_pronostico(train_d, test_d, pron_xgb_d, 'Diaria - XGBoost')

In [0]:
pron_lstm_d, mod_lstm_d = ajustar_lstm(train_d, horizonte=len(test_d), indice_pronostico=test_d.index, ventana=30, epochs=40, unidades=50)
print(registrar_resultado('Diaria', 'LSTM', test_d.values, pron_lstm_d.values, total_real=total_dic_2017))
registro_modelos[('Diaria', 'LSTM')] = dict(fn=ajustar_lstm, kwargs=dict(ventana=30, epochs=40, unidades=50))
graficar_pronostico(train_d, test_d, pron_lstm_d, 'Diaria - LSTM')

**Observaciones - granularidad diaria:** es esperable que Prophet obtenga el mejor desempeño en esta granularidad al capturar simultáneamente la estacionalidad semanal y anual, mientras que Holt-Winters y SARIMA (limitados a `m=7`) deberían mostrar mayor error al no anticipar la caída de ventas de diciembre. XGBoost, al incorporar variables de fecha explícitas, debería comportarse razonablemente. LSTM, sin variables de fecha y con una ventana (30 días) mucho más corta que el ciclo anual, es el que más debería sufrir para anticipar el patrón estacional de largo plazo. Los números exactos se revisan en la tabla comparativa de la sección 11.

## 9. Modelado - Granularidad semanal

Configuración usada en esta granularidad:
- **Holt-Winters:** `m=52` (estacionalidad anual en semanas).
- **SARIMA:** se probó inicialmente con `m=52`, pero **resultó computacionalmente inviable** (el ajuste no llega a converger en un tiempo razonable y puede agotar la memoria disponible al construir el filtro de Kalman con un periodo estacional tan largo - un problema conocido de `statsmodels`/SARIMAX con periodos estacionales grandes). Por eso se usa **ARIMA no estacional** para esta granularidad, dejando que Holt-Winters y Prophet capturen el patrón anual a nivel semanal. Esta limitación práctica se documenta aquí como parte del análisis pedido en el enunciado.
- **Prophet:** `freq='W'`.
- **XGBoost:** 4 rezagos (aprox. un mes de historia semanal).
- **LSTM:** ventana de 52 semanas (un ciclo anual completo), para darle al modelo la oportunidad de "ver" un año completo hacia atrás.

In [0]:
pron_hw_w, mod_hw_w = ajustar_holt_winters(train_w, horizonte=len(test_w), indice_pronostico=test_w.index, seasonal_periods=52)
print(registrar_resultado('Semanal', 'Holt-Winters', test_w.values, pron_hw_w.values, total_real=total_dic_2017))
registro_modelos[('Semanal', 'Holt-Winters')] = dict(fn=ajustar_holt_winters, kwargs=dict(seasonal_periods=52))
graficar_pronostico(train_w, test_w, pron_hw_w, 'Semanal - Holt-Winters', contexto=30)

In [0]:
pron_arima_w, mod_arima_w = ajustar_auto_arima(train_w, horizonte=len(test_w), indice_pronostico=test_w.index, seasonal_periods=None)
print(registrar_resultado('Semanal', 'ARIMA (no estacional)', test_w.values, pron_arima_w.values, total_real=total_dic_2017))
registro_modelos[('Semanal', 'ARIMA (no estacional)')] = dict(fn=ajustar_auto_arima, kwargs=dict(seasonal_periods=None))
graficar_pronostico(train_w, test_w, pron_arima_w, 'Semanal - ARIMA (no estacional)', contexto=30)

In [0]:
pron_prophet_w, mod_prophet_w = ajustar_prophet(train_w, horizonte=len(test_w), indice_pronostico=test_w.index, freq='W')
print(registrar_resultado('Semanal', 'Prophet', test_w.values, pron_prophet_w.values, total_real=total_dic_2017))
registro_modelos[('Semanal', 'Prophet')] = dict(fn=ajustar_prophet, kwargs=dict(freq='W'))
graficar_pronostico(train_w, test_w, pron_prophet_w, 'Semanal - Prophet', contexto=30)

In [0]:
pron_xgb_w, mod_xgb_w = ajustar_xgboost(train_w, horizonte=len(test_w), indice_pronostico=test_w.index, n_lags=4)
print(registrar_resultado('Semanal', 'XGBoost', test_w.values, pron_xgb_w.values, total_real=total_dic_2017))
registro_modelos[('Semanal', 'XGBoost')] = dict(fn=ajustar_xgboost, kwargs=dict(n_lags=4))
graficar_pronostico(train_w, test_w, pron_xgb_w, 'Semanal - XGBoost', contexto=30)

In [0]:
pron_lstm_w, mod_lstm_w = ajustar_lstm(train_w, horizonte=len(test_w), indice_pronostico=test_w.index, ventana=52, epochs=80, unidades=32)
print(registrar_resultado('Semanal', 'LSTM', test_w.values, pron_lstm_w.values, total_real=total_dic_2017))
registro_modelos[('Semanal', 'LSTM')] = dict(fn=ajustar_lstm, kwargs=dict(ventana=52, epochs=80, unidades=32))
graficar_pronostico(train_w, test_w, pron_lstm_w, 'Semanal - LSTM', contexto=30)

**Observaciones - granularidad semanal:** a este nivel se reduce el ruido diario y Holt-Winters (que sí logra manejar `m=52`) debería recuperar buena parte de su capacidad de capturar la estacionalidad anual. El ARIMA no estacional, al no tener forma de anticipar el patrón anual, debería comportarse de forma similar a una caminata aleatoria (proyección casi plana). LSTM, pese a usar una ventana de un año completo, cuenta con relativamente pocos ciclos anuales completos de entrenamiento (~5), lo que típicamente limita su capacidad de generalizar el patrón estacional frente a métodos clásicos que lo incorporan explícitamente en su estructura.

## 10. Modelado - Granularidad mensual

Configuración usada en esta granularidad:
- **Holt-Winters / SARIMA:** `m=12` (estacionalidad anual en meses), igual que en el caso visto en clase.
- **Prophet:** `freq='MS'`.
- **XGBoost:** 3 rezagos.
- **LSTM:** ventana de 12 meses.

**Advertencia:** a nivel mensual solo hay 59 observaciones de entrenamiento (menos de 5 años). Esto es consistente con los casos vistos en clase, pero es un volumen de datos reducido para modelos que necesitan aprender patrones desde los datos (XGBoost, LSTM), y favorece a los modelos clásicos que incorporan la estacionalidad como supuesto estructural (Holt-Winters, SARIMA, Prophet).

In [0]:
pron_hw_m, mod_hw_m = ajustar_holt_winters(train_m, horizonte=len(test_m), indice_pronostico=test_m.index, seasonal_periods=12)
print(registrar_resultado('Mensual', 'Holt-Winters', test_m.values, pron_hw_m.values, total_real=total_dic_2017))
registro_modelos[('Mensual', 'Holt-Winters')] = dict(fn=ajustar_holt_winters, kwargs=dict(seasonal_periods=12))
graficar_pronostico(train_m, test_m, pron_hw_m, 'Mensual - Holt-Winters', contexto=24)

In [0]:
pron_arima_m, mod_arima_m = ajustar_auto_arima(train_m, horizonte=len(test_m), indice_pronostico=test_m.index, seasonal_periods=12)
print(registrar_resultado('Mensual', 'SARIMA (auto_arima)', test_m.values, pron_arima_m.values, total_real=total_dic_2017))
registro_modelos[('Mensual', 'SARIMA (auto_arima)')] = dict(fn=ajustar_auto_arima, kwargs=dict(seasonal_periods=12))
graficar_pronostico(train_m, test_m, pron_arima_m, 'Mensual - SARIMA (auto_arima)', contexto=24)

In [0]:
pron_prophet_m, mod_prophet_m = ajustar_prophet(train_m, horizonte=len(test_m), indice_pronostico=test_m.index, freq='MS')
print(registrar_resultado('Mensual', 'Prophet', test_m.values, pron_prophet_m.values, total_real=total_dic_2017))
registro_modelos[('Mensual', 'Prophet')] = dict(fn=ajustar_prophet, kwargs=dict(freq='MS'))
graficar_pronostico(train_m, test_m, pron_prophet_m, 'Mensual - Prophet', contexto=24)

In [0]:
pron_xgb_m, mod_xgb_m = ajustar_xgboost(train_m, horizonte=len(test_m), indice_pronostico=test_m.index, n_lags=3)
print(registrar_resultado('Mensual', 'XGBoost', test_m.values, pron_xgb_m.values, total_real=total_dic_2017))
registro_modelos[('Mensual', 'XGBoost')] = dict(fn=ajustar_xgboost, kwargs=dict(n_lags=3))
graficar_pronostico(train_m, test_m, pron_xgb_m, 'Mensual - XGBoost', contexto=24)

In [0]:
pron_lstm_m, mod_lstm_m = ajustar_lstm(train_m, horizonte=len(test_m), indice_pronostico=test_m.index, ventana=12, epochs=80, unidades=32)
print(registrar_resultado('Mensual', 'LSTM', test_m.values, pron_lstm_m.values, total_real=total_dic_2017))
registro_modelos[('Mensual', 'LSTM')] = dict(fn=ajustar_lstm, kwargs=dict(ventana=12, epochs=80, unidades=32))
graficar_pronostico(train_m, test_m, pron_lstm_m, 'Mensual - LSTM', contexto=24)

**Observaciones - granularidad mensual:** con una serie tan corta, es esperable que los modelos clásicos con estacionalidad estructural (Holt-Winters, SARIMA) y Prophet sean los más precisos, mientras que XGBoost y LSTM, al depender más de los datos para "aprender" el patrón, probablemente muestren mayor error relativo - de forma similar a lo observado en el caso de comparación visto en clase. Nota: con un único punto de prueba (`test_m` tiene 1 periodo), el `r2_score` no está definido matemáticamente (queda como `NaN`); las demás métricas (RMSE, MAPE, error % del total) sí son interpretables.

## 11. Comparación final de resultados

In [0]:
resultados_df = pd.DataFrame(resultados)
orden_cols = ['granularidad', 'modelo', 'rmse', 'mean_absolute_error', 'mape', 'r2_score',
              'total_predicho_mes', 'total_real_mes', 'error_pct_total_mes',
              'median_absolute_error', 'mse', 'msle']
orden_cols = [c for c in orden_cols if c in resultados_df.columns]
resultados_df = resultados_df[orden_cols].sort_values('rmse').reset_index(drop=True)
resultados_df

In [0]:
pivote_mape = resultados_df.pivot(index='modelo', columns='granularidad', values='mape')
pivote_mape.plot(kind='bar', figsize=(11, 5))
plt.title('MAPE (%) por modelo y granularidad - backtest diciembre 2017')
plt.ylabel('MAPE (%)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Granularidad')
plt.tight_layout(); plt.show()

In [0]:
ranking_final = resultados_df.reindex(resultados_df['error_pct_total_mes'].abs().sort_values().index).reset_index(drop=True)
mejor = ranking_final.iloc[0]

print("MEJOR COMBINACION (menor error % sobre el total real de diciembre 2017):")
print(f"  Granularidad: {mejor['granularidad']}")
print(f"  Modelo:       {mejor['modelo']}")
print(f"  Error % en el total del mes: {mejor['error_pct_total_mes']:+.2f}%")
print(f"  RMSE (escala nativa):        {mejor['rmse']:,.1f}")
print(f"  MAPE (escala nativa):        {mejor['mape']:.2f}%")
print()
print("Top 3 combinaciones por este criterio:")
ranking_final[['granularidad', 'modelo', 'error_pct_total_mes', 'rmse', 'mape']].head(3)

**Lectura de la tabla comparativa:**
- Las columnas `rmse` / `mape` comparan el ajuste punto a punto en la escala nativa de cada granularidad (no son directamente comparables entre granularidades, ya que un día y un mes tienen escalas de venta muy distintas).
- La columna `error_pct_total_mes` sí es comparable entre granularidades: mide qué tan cerca quedó el **total de diciembre 2017 pronosticado** (sumando el horizonte correspondiente) frente al total real (695,170 unidades), que es en el fondo la pregunta que hace el enunciado ("evaluar si es mejor considerar tiempos diarios, semanales o mensuales" para predecir el total del próximo mes).
- Con base en ambas lecturas se elige la combinación con menor error como referencia principal, pero conviene revisar también el segundo y tercer lugar para verificar que la conclusión sea razonablemente robusta y no dependa de un único resultado.

In [0]:
resultados_df.reindex(resultados_df['error_pct_total_mes'].abs().sort_values().index).head(3)

## 12. Pronóstico genuino de ventas totales - Enero 2018

Con la metodología ya validada contra diciembre de 2017, se reentrena la mejor combinación (y las siguientes 2 mejores, para contrastar) usando **toda la información disponible** (incluyendo diciembre de 2017) y se genera el pronóstico real para **enero de 2018**, que es el "próximo mes" pedido en el enunciado.

In [0]:
def construir_indice_pronostico(freq, ultima_fecha, inicio_objetivo, fin_objetivo, max_periodos=40):
    inicio_objetivo = pd.Timestamp(inicio_objetivo)
    fin_objetivo = pd.Timestamp(fin_objetivo)
    if freq == 'D':
        return pd.date_range(inicio_objetivo, fin_objetivo, freq='D')
    if freq == 'MS':
        return pd.date_range(inicio_objetivo, fin_objetivo, freq='MS')
    if freq == 'W':
        candidatas = pd.date_range(start=ultima_fecha, periods=max_periodos, freq='W-SUN')
        return candidatas[(candidatas >= inicio_objetivo) & (candidatas <= fin_objetivo)]
    raise ValueError(freq)


series_completas = {'Diaria': serie_diaria, 'Semanal': serie_semanal, 'Mensual': serie_mensual}
freqs_grano = {'Diaria': 'D', 'Semanal': 'W', 'Mensual': 'MS'}

top3 = resultados_df.reindex(resultados_df['error_pct_total_mes'].abs().sort_values().index).head(3)

pronosticos_enero = []
for _, fila in top3.iterrows():
    grano, modelo_nombre = fila['granularidad'], fila['modelo']
    serie_completa = series_completas[grano]
    freq = freqs_grano[grano]
    indice_enero = construir_indice_pronostico(freq, serie_completa.index.max(), '2018-01-01', '2018-01-31')

    config = registro_modelos[(grano, modelo_nombre)]
    pronostico, _ = config['fn'](serie_completa, horizonte=len(indice_enero),
                                  indice_pronostico=indice_enero, **config['kwargs'])

    total_enero = float(np.clip(pronostico.values, 0, None).sum())
    pronosticos_enero.append({
        'granularidad': grano, 'modelo': modelo_nombre,
        'periodos_pronosticados': len(indice_enero),
        'total_predicho_enero_2018': total_enero
    })
    print(f"{grano} / {modelo_nombre}: total predicho enero 2018 = {total_enero:,.0f} ({len(indice_enero)} periodos)")

pronosticos_enero_df = pd.DataFrame(pronosticos_enero)
pronosticos_enero_df

La primera fila de la tabla anterior (`top3`, ordenada por menor error absoluto en el backtest de diciembre) corresponde a la combinación recomendada como respuesta principal a la pregunta del enunciado. Las otras dos se muestran como contraste, para verificar que el pronóstico de enero 2018 sea razonablemente consistente entre métodos y no dependa de un único modelo.

*Nota:* para la granularidad semanal, el criterio de "semana cuyo cierre cae en el mes" cubre solo 4 de las 5 semanas de enero 2018 (los últimos ~3 días del mes quedan en la semana que cierra el 4 de febrero); por eso el total semanal de enero puede quedar levemente subestimado frente a los enfoques diario y mensual. Esto se documenta como la misma simplificación de borde ya mencionada en la sección 6.

## 13. Conclusiones

In [0]:
pronostico_ganador = pronosticos_enero_df[
    (pronosticos_enero_df['granularidad'] == mejor['granularidad']) &
    (pronosticos_enero_df['modelo'] == mejor['modelo'])
].iloc[0]

print("=" * 72)
print("VEREDICTO FINAL")
print("=" * 72)
print(f"Mejor combinacion: {mejor['modelo']}  |  Granularidad: {mejor['granularidad']}")
print(f"Justificacion: menor error % sobre el total real de diciembre 2017 en el backtest "
      f"({mejor['error_pct_total_mes']:+.2f}%)")
print()
print(f">>> PRONOSTICO DE VENTAS TOTALES PARA ENERO 2018: {pronostico_ganador['total_predicho_enero_2018']:,.0f} unidades <<<")
print()
print("Contraste con las otras 2 combinaciones evaluadas en la seccion 12 (para ver que tan robusto es el resultado):")
print(pronosticos_enero_df.to_string(index=False))

**Sobre la granularidad (diaria vs. semanal vs. mensual):** no hay una respuesta única válida para cualquier problema; depende de cómo interactúan el ruido de la serie, el número de estacionalidades presentes y la cantidad de historia disponible:
- La granularidad **diaria** ofrece más datos de entrenamiento y le permite a modelos como Prophet capturar tanto el patrón semanal como el anual, pero es más ruidosa y penaliza a los modelos que solo soportan una estacionalidad (Holt-Winters, SARIMA con `m=7`).
- La granularidad **semanal** reduce el ruido diario y le devuelve a Holt-Winters su capacidad de modelar la estacionalidad anual (`m=52`), pero para SARIMA resultó computacionalmente inviable con ese periodo estacional.
- La granularidad **mensual** es la más limpia y estable, pero al tener solo 59 observaciones limita a los modelos que dependen más de los datos (XGBoost, LSTM).

**Sobre los modelos:** los modelos clásicos con estacionalidad explícita (Holt-Winters, SARIMA, Prophet) tienden a tener ventaja cuando la historia disponible es corta o cuando el periodo estacional es difícil de aprender solo de los datos. Prophet tiene la ventaja adicional de poder combinar múltiples estacionalidades a la vez, lo que lo hace especialmente competitivo a nivel diario. Los modelos que dependen más de aprender patrones desde los datos (XGBoost, LSTM) parecen requerir más historia y/o variables explícitas de estacionalidad (como las de fecha que se le dieron a XGBoost) para ser competitivos en este problema. La recomendación final debe leerse directamente de la tabla de la sección 11, generada al ejecutar este notebook.

**Limitaciones y posibles extensiones:**
- El backtest se valida contra un único mes (diciembre 2017); una validación cruzada sobre varios meses daría una comparación más robusta.
- No se incorporaron variables externas (feriados, promociones, precios) que podrían mejorar cualquiera de los modelos.
- El análisis se hizo sobre la serie agregada total; `store` e `item` se usaron solo para caracterizar el negocio en el EDA. Un siguiente paso razonable sería repetir el mejor enfoque a nivel de tienda o de categoría de producto, si el negocio lo requiere.
- SARIMA con periodo estacional largo (`m=52`) no fue viable computacionalmente con `statsmodels`/`pmdarima` en este entorno; una alternativa a explorar sería SARIMAX con términos de Fourier como regresores exógenos para aproximar la estacionalidad anual sin pagar el costo computacional de un `m` tan grande.